In [ ]:
import matplotlib
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plots
import numpy as np
plots.style.use('fivethirtyeight')

## New material

**Reminder:** We are treating this dataset as a random sample from a hosptial system.

In [ ]:
births = Table.read_table('baby.csv')
births.show(6) 

### Converting to standard units

In [ ]:
def standard_units(x):
    """Convert array x to standard units."""
    return (x - np.mean(x)) / np.std(x)

In [ ]:
ages = births.column('Maternal Age')
ages

In [ ]:
ages_standard_units = standard_units(ages)
ages_standard_units

In [ ]:
print('Mean in SU:', np.mean(ages_standard_units))
print('SD in SU:', np.std(ages_standard_units))

#### How does a distribution change when converted to standard units?

In [ ]:
both = Table().with_columns(
    'Age in Years', ages,
    'Age in Standard Units', ages_standard_units
)
both

In [ ]:
both.hist('Age in Years', bins = np.arange(15, 46, 2))

In [ ]:
np.mean(ages), np.std(ages)

In [ ]:
both.hist('Age in Standard Units', bins = np.arange(-2.2, 3.4, 0.35))
plots.xlim(-2, 3.1);

In [ ]:
births = Table.read_table('baby.csv')
births.show(6)

### A bell shape tells us more about a distribution

In [ ]:
births.hist('Birth Weight')

In [ ]:
def chebyshev(num_SDs):
    # returns the least proportion of the data in +/- num_SDs
    z = num_SDs
    return 1 - 1/z**2  

**Task**: Empirically verify that for a bell shaped distribution,
- the proportion of values within 1, 2 and 3 SDs of the mean is 68, 95, and 99.75 percent, respectively.

In [ ]:
birth_weight_mean = np.average(births.column('Birth Weight'))
birth_weight_mean

In [ ]:
birth_weight_sd = np.std(births.column('Birth Weight'))
birth_weight_sd

###### Within 1 SD

In [ ]:
chebyshev(1)

In [ ]:
births.where('Birth Weight', are.between(birth_weight_mean - birth_weight_sd,
                                        birth_weight_mean + birth_weight_sd)).num_rows/births.num_rows

###### Within 2 SD

In [ ]:
chebyshev(2)

In [ ]:
births.where('Birth Weight', are.between(birth_weight_mean - 2*birth_weight_sd,
                                        birth_weight_mean + 2*birth_weight_sd)).num_rows/births.num_rows

###### Within 3 SD

In [ ]:
chebyshev(3)

In [ ]:
births.where('Birth Weight', are.between(birth_weight_mean - 3*birth_weight_sd,
                                        birth_weight_mean + 3*birth_weight_sd)).num_rows/births.num_rows

**STOP**

### The Central Limit Theorem (CLT)

In [ ]:
united = Table.read_table('united.csv')

In [ ]:
united.show(6)

**Reminder:** We are treating this dataset as the population of all United Airlines flights that occurred over the two month period.

In [ ]:
population_parameter = np.mean(united.column('Delay'))
population_parameter

**Task**: Show that the empirical distribution of the sample mean approaches a bell shape as the sample size (for each statistic used to create the distribution) increases.

- Though the CLT is a statement about the probability distribution of the statistic, the empirical distribution will look roughly like the probability distribution when the repetition size is large. Here, we set it to 10,000.

In [ ]:
def one_sample_mean(sample_size):
    """ 
    Takes a sample from the population of flights 
    and computes its mean
    """
    sampled_flights = united.sample(sample_size)
    return np.mean(sampled_flights.column('Delay'))

In [ ]:
def ten_thousand_sample_means(sample_size):
    means = make_array()
    for i in np.arange(10000):
        mean = one_sample_mean(sample_size)
        means = np.append(means, mean)
    return means

In [ ]:
def plot_sample_mean_distribution(sample_size):
    sample_means = ten_thousand_sample_means(sample_size)
    
    Table().with_column('Mean of ' + str(sample_size) + ' flight delays', 
                        sample_means).hist(bins=20)

    print('Population Average:', population_parameter)

In [ ]:
plot_sample_mean_distribution(10)

In [ ]:
plot_sample_mean_distribution(30)

In [ ]:
plot_sample_mean_distribution(100)

In [ ]:
plot_sample_mean_distribution(1000)

## Discussion Question

After rolling 1,000,000 fair 6-sided dice, which of these histograms would you expect to have a bell shape? 

###### Histogram 1: The histogram of outcomes of these one million rolls

In [ ]:
die = make_array(1,2,3,4,5,6)

In [ ]:
one_million_die_rolls = np.random.choice(die,size = 1000000)

In [ ]:
Table().with_columns('Die Spot', one_million_die_rolls).hist(bins = np.arange(1,8))

###### Histogram 2: The histogram that results from computing the average outcome of these one million rolls

In [ ]:
one_million_die_average = np.average(one_million_die_rolls)
one_million_die_average

In [ ]:
Table().with_columns('Average', one_million_die_average).hist(bins = np.arange(1,8))
plots.title('This is not much of a histogram!');

###### Histogram 3: The histogram that results from:
 - ###### splitting the outcomes into 1,000 groups of 1,000 (in the order they occurred)
 - ###### and computing the average outcome of each group

In [ ]:
one_million_roll_table = Table().with_columns('Die Spot', one_million_die_rolls)

In [ ]:
np.average(one_million_roll_table.take(np.arange(1000)).column('Die Spot'))

In [ ]:
group_averages = make_array()
start = 0
for i in np.arange(1000):
    group_average = np.average(one_million_roll_table.take(np.arange(start, start + 1000)).column('Die Spot'))
    group_averages = np.append(group_averages, group_average)
    start = start + 1000

In [ ]:
group_averages

In [ ]:
Table().with_columns('Average', group_averages).hist()